In [ ]:
# PART 2 — Customer Complaints Clustering

import re
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from nltk.stem import PorterStemmer
from tabulate import tabulate

df = pd.read_csv("customer_complaints_1.csv")

stemmer = PorterStemmer()

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    words = text.split()
    words = [stemmer.stem(w) for w in words if w not in ENGLISH_STOP_WORDS]
    return " ".join(words)

df["clean_text"] = df["text"].apply(preprocess)


In [ ]:
vectorizer = TfidfVectorizer(max_df=0.9)
X = vectorizer.fit_transform(df["clean_text"])

k = 4
km = KMeans(n_clusters=k, random_state=42, n_init=20)
df["cluster"] = km.fit_predict(X)

table = [["No", "Cluster", "Text"]]
for i, row in df.iterrows():
    table.append([i+1, row["cluster"], row["text"][:100] + "..."])

print(tabulate(table, headers="firstrow"))


In [ ]:
print("\nTop terms per cluster:")
terms = vectorizer.get_feature_names_out()
order = km.cluster_centers_.argsort()[:, ::-1]

for i in range(k):
    print(f"\nCluster {i}:")
    for ind in order[i, :10]:
        print(terms[ind])

df.to_csv("clustered_complaints.csv", index=False)
